# ============================================================================
# ANÁLISE ESTATÍSTICA COMPLETA - HOMICÍDIOS BRASIL E MUNDO
# ============================================================================
# Autor: Projeto de Ciência de Dados
# Descrição: Análise exploratória, testes estatísticos e modelos preditivos

### Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, ttest_ind, mannwhitneyu, f_oneway, kruskal
from scipy.stats import pearsonr, spearmanr, linregress
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

### Configurações estéticas

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("="*70)
print("ANÁLISE ESTATÍSTICA COMPLETA - HOMICÍDIOS BRASIL E MUNDO")
print("="*70)

### 1. CARREGAMENTO DOS DADOS

In [ ]:
print("\n📂 CARREGANDO DADOS...")

try:
    df_estados = pd.read_csv('dados/homicidios_brasil_estados_1999_2026.csv')
    df_regioes = pd.read_csv('dados/homicidios_brasil_regioes_1999_2026.csv')
    df_brasil = pd.read_csv('dados/homicidios_brasil_nacional_1999_2026.csv')
    df_mundo = pd.read_csv('dados/homicidios_mundo_paises_1999_2026.csv')
    df_comparacao = pd.read_csv('dados/comparacao_brasil_mundo_1999_2026.csv')
    print("✅ Dados carregados com sucesso!")
except FileNotFoundError:
    print("❌ Arquivos não encontrados! Execute primeiro o código de geração.")
    exit()

### 2. ANÁLISE DESCRITIVA COMPLETA

In [ ]:
print("\n" + "="*70)
print("2. ANÁLISE DESCRITIVA")
print("="*70)

print("\n📊 ESTATÍSTICAS DESCRITIVAS - TAXA DE HOMICÍDIOS BRASIL (2000-2024):")
dados_historicos = df_brasil[(df_brasil['ano'] >= 2000) & (df_brasil['ano'] <= 2024)]
print(dados_historicos['taxa_homicidios'].describe())

print("\n📊 ESTATÍSTICAS DESCRITIVAS - HOMICÍDIOS POR ESTADO (2024):")
dados_2024 = df_estados[df_estados['ano'] == 2024]
print(dados_2024['taxa_homicidios'].describe())

print("\n📊 ESTATÍSTICAS DESCRITIVAS - COMPARAÇÃO MUNDIAL (2024):")
dados_mundo_2024 = df_mundo[df_mundo['ano'] == 2024]
print(dados_mundo_2024['taxa_homicidios'].describe())

# Estatísticas por região
print("\n📊 MÉDIA DA TAXA POR REGIÃO (2024):")
media_regiao = df_estados[df_estados['ano'] == 2024].groupby('regiao')['taxa_homicidios'].agg(['mean', 'std', 'min', 'max']).round(1)
print(media_regiao)

### 3. ANÁLISE DE TENDÊNCIA TEMPORAL

In [ ]:
print("\n" + "="*70)
print("3. ANÁLISE DE TENDÊNCIA TEMPORAL")
print("="*70)

# Teste de tendência usando correlação de Spearman (não paramétrica)
anos_analise = df_brasil['ano'].values
taxas_brasil = df_brasil['taxa_homicidios'].values

correlacao, p_valor = spearmanr(anos_analise, taxas_brasil)
print(f"\n📈 CORRELAÇÃO DE SPEARMAN (Ano vs Taxa Brasil):")
print(f"  Coeficiente: {correlacao:.3f}")
print(f"  P-valor: {p_valor:.4f}")
if p_valor < 0.05:
    print(f"  ✅ Tendência significativa (p < 0.05)")
    if correlacao > 0:
        print(f"  📈 Tendência de AUMENTO ao longo do tempo")
    else:
        print(f"  📉 Tendência de DIMINUIÇÃO ao longo do tempo")
else:
    print(f"  ❌ Sem tendência significativa")

# Análise de quebra estrutural (mudança de padrão)
print("\n🔍 ANÁLISE DE PONTOS DE MUDANÇA (Breakpoints):")
anos_pontos = [2002, 2008, 2014, 2018, 2022]
for ano_ponto in anos_pontos:
    antes = df_brasil[df_brasil['ano'] < ano_ponto]['taxa_homicidios'].mean()
    depois = df_brasil[df_brasil['ano'] >= ano_ponto]['taxa_homicidios'].mean()
    variacao = ((depois - antes) / antes) * 100
    print(f"  {ano_ponto}: Antes={antes:.1f} → Depois={depois:.1f} (Δ {variacao:+.1f}%)")

# Decomposição de série temporal
print("\n📉 DECOMPOSIÇÃO DA SÉRIE TEMPORAL:")
serie_brasil = df_brasil.set_index('ano')['taxa_homicidios']
try:
    decomposicao = seasonal_decompose(serie_brasil, model='additive', period=5)
    print("  ✅ Decomposição realizada com sucesso!")
    print(f"  Tendência geral: {'Crescente' if decomposicao.trend.iloc[-1] > decomposicao.trend.iloc[0] else 'Decrescente'}")
except:
    print("  ⚠️ Série muito curta para decomposição sazonal")

### 4. TESTES DE NORMALIDADE

In [ ]:
print("\n" + "="*70)
print("4. TESTES DE NORMALIDADE")
print("="*70)

# Shapiro-Wilk para distribuição das taxas em 2024
dados_test = df_estados[df_estados['ano'] == 2024]['taxa_homicidios']
stat, p_value = shapiro(dados_test)

print(f"\n📊 TESTE DE SHAPIRO-WILK (Taxas 2024):")
print(f"  Estatística: {stat:.4f}")
print(f"  P-valor: {p_value:.4f}")
if p_value > 0.05:
    print("  ✅ Distribuição NORMAL (não rejeita H0)")
else:
    print("  ❌ Distribuição NÃO-NORMAL (rejeita H0)")

# Teste de normalidade para comparação mundial
dados_mundo_2024_test = df_mundo[df_mundo['ano'] == 2024]['taxa_homicidios']
stat_m, p_value_m = shapiro(dados_mundo_2024_test)
print(f"\n📊 TESTE DE SHAPIRO-WILK (Taxas Mundo 2024):")
print(f"  Estatística: {stat_m:.4f}")
print(f"  P-valor: {p_value_m:.4f}")
if p_value_m > 0.05:
    print("  ✅ Distribuição NORMAL")
else:
    print("  ❌ Distribuição NÃO-NORMAL")

### 5. ANÁLISE DE VARIÂNCIA (ANOVA)

In [ ]:
print("\n" + "="*70)
print("5. ANÁLISE DE VARIÂNCIA (ANOVA)")
print("="*70)

# ANOVA para comparar regiões em 2024
regioes = df_estados[df_estados['ano'] == 2024]['regiao'].unique()
grupos = [df_estados[(df_estados['ano'] == 2024) & (df_estados['regiao'] == r)]['taxa_homicidios'] for r in regioes]

# Teste paramétrico (ANOVA)
f_stat, p_anova = f_oneway(*grupos)
print(f"\n📊 ANOVA (Comparação entre Regiões 2024):")
print(f"  Estatística F: {f_stat:.4f}")
print(f"  P-valor: {p_anova:.4f}")
if p_anova < 0.05:
    print("  ✅ Diferença SIGNIFICATIVA entre regiões")
    print("  📍 As regiões têm perfis de violência distintos")
else:
    print("  ❌ Sem diferença significativa entre regiões")

# Teste não-paramétrico (Kruskal-Wallis) como complemento
h_stat, p_kruskal = kruskal(*grupos)
print(f"\n📊 KRUSKAL-WALLIS (Teste não-paramétrico):")
print(f"  Estatística H: {h_stat:.4f}")
print(f"  P-valor: {p_kruskal:.4f}")
if p_kruskal < 0.05:
    print("  ✅ Confirma diferença significativa entre regiões")

### 6. CORRELAÇÕES E RELACIONAMENTOS

In [ ]:
print("\n" + "="*70)
print("6. CORRELAÇÕES E RELACIONAMENTOS")
print("="*70)

# Correlação entre população e homicídios
df_2024 = df_estados[df_estados['ano'] == 2024]
corr_pop, p_pop = pearsonr(df_2024['populacao'], df_2024['homicidios'])
corr_pop_taxa, p_pop_taxa = pearsonr(df_2024['populacao'], df_2024['taxa_homicidios'])

print(f"\n📊 CORRELAÇÃO (2024):")
print(f"  População vs Homicídios absolutos: r={corr_pop:.3f} (p={p_pop:.4f})")
print(f"  População vs Taxa por 100k: r={corr_pop_taxa:.3f} (p={p_pop_taxa:.4f})")

# Matriz de correlação para séries temporais
print("\n📊 MATRIZ DE CORRELAÇÃO (Séries Temporais):")
df_corr = df_estados.pivot(index='ano', columns='sigla', values='taxa_homicidios')
matriz_corr = df_corr.corr()
print(f"  Média de correlação entre estados: {matriz_corr.mean().mean():.3f}")
print(f"  Correlação máxima: {matriz_corr.max().max():.3f}")
print(f"  Correlação mínima: {matriz_corr.min().min():.3f}")

# Estados mais correlacionados
print("\n🔗 ESTADOS MAIS CORRELACIONADOS (Similaridade nas tendências):")
corr_flat = matriz_corr.unstack().sort_values(ascending=False)
corr_flat = corr_flat[corr_flat < 1]  # Remover diagonal
top_corrs = corr_flat.head(5)
for (estado1, estado2), corr in top_corrs.items():
    print(f"  {estado1} ↔ {estado2}: r={corr:.3f}")

### 7. COMPARAÇÃO BRASIL VS MUNDO (TESTES ESTATÍSTICOS)

In [ ]:
print("\n" + "="*70)
print("7. COMPARAÇÃO BRASIL VS MUNDO")
print("="*70)

# Dados para comparação (2024)
taxa_brasil_2024 = df_brasil[df_brasil['ano'] == 2024]['taxa_homicidios'].values[0]
taxas_mundo_2024 = df_mundo[df_mundo['ano'] == 2024]['taxa_homicidios'].values
taxas_mundo_sem_brasil = taxas_mundo_2024[df_mundo[df_mundo['ano'] == 2024]['codigo'] != 'BRA']

# Teste t para comparar Brasil com média mundial
media_mundo = taxas_mundo_sem_brasil.mean()
t_stat, p_t = ttest_ind([taxa_brasil_2024]*len(taxas_mundo_sem_brasil), taxas_mundo_sem_brasil)

print(f"\n📊 COMPARAÇÃO BRASIL vs MUNDO (2024):")
print(f"  Taxa Brasil: {taxa_brasil_2024:.1f} / 100k")
print(f"  Média Mundial (sem Brasil): {media_mundo:.1f} / 100k")
print(f"  Diferença: {taxa_brasil_2024 - media_mundo:+.1f}")
print(f"  Brasil é {taxa_brasil_2024/media_mundo:.1f}x maior que média mundial")

# Posição do Brasil no ranking mundial
ranking_mundo = df_mundo[df_mundo['ano'] == 2024].sort_values('taxa_homicidios', ascending=False)
posicao_brasil = ranking_mundo[ranking_mundo['codigo'] == 'BRA'].index[0] + 1
total_paises = len(ranking_mundo)
percentil = (1 - posicao_brasil/total_paises) * 100

print(f"\n📊 POSIÇÃO DO BRASIL NO RANKING MUNDIAL (2024):")
print(f"  Posição: {posicao_brasil}º de {total_paises} países")
print(f"  Percentil: {percentil:.1f}% (mais violento)")
print(f"  País mais violento: {ranking_mundo.iloc[0]['pais']} ({ranking_mundo.iloc[0]['taxa_homicidios']:.1f})")
print(f"  País menos violento: {ranking_mundo.iloc[-1]['pais']} ({ranking_mundo.iloc[-1]['taxa_homicidios']:.1f})")

# Teste Mann-Whitney para comparar distribuições
mw_stat, p_mw = mannwhitneyu([taxa_brasil_2024]*len(taxas_mundo_sem_brasil), taxas_mundo_sem_brasil)
print(f"\n📊 TESTE MANN-WHITNEY (Brasil vs Mundo):")
print(f"  Estatística U: {mw_stat:.1f}")
print(f"  P-valor: {p_mw:.4f}")
if p_mw < 0.05:
    print("  ✅ Brasil é estatisticamente DIFERENTE do mundo")
else:
    print("  ❌ Brasil não difere estatisticamente do mundo")

### 8. ANÁLISE DE CLUSTER (AGRUPAMENTO DE ESTADOS)

In [ ]:
print("\n" + "="*70)
print("8. ANÁLISE DE CLUSTER (Agrupamento de Estados)")
print("="*70)

# Preparar dados para clustering
df_cluster = df_estados[df_estados['ano'] == 2024][['sigla', 'taxa_homicidios', 'populacao']]
X = df_cluster[['taxa_homicidios', 'populacao']].values

# Padronizar dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Determinar número ótimo de clusters (Elbow Method)
inertias = []
K_range = range(1, 8)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

print(f"\n📊 MÉTODO DO COTOVELO (Elbow Method):")
for k, inercia in zip(K_range, inertias):
    print(f"  K={k}: Inércia={inercia:.0f}")

# Aplicar K-Means com k=4 (baseado nos dados)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\n📊 DISTRIBUIÇÃO DOS CLUSTERS:")
for i in range(4):
    cluster_data = df_cluster[df_cluster['cluster'] == i]
    print(f"\nCluster {i+1}: {len(cluster_data)} estados")
    print(f"  Taxa média: {cluster_data['taxa_homicidios'].mean():.1f}")
    print(f"  Pop. média: {cluster_data['populacao'].mean()/1000000:.1f}M")
    print(f"  Estados: {', '.join(cluster_data['sigla'].tolist())}")

### 9. ANÁLISE PREDITIVA (PREVISÃO DE CURTO PRAZO)

In [ ]:
print("\n" + "="*70)
print("9. ANÁLISE PREDITIVA")
print("="*70)

# Previsão para 2025-2026 usando ARIMA
print("\n📈 PREVISÃO ARIMA (Modelo de Séries Temporais):")

# Preparar série
serie = df_brasil.set_index('ano')['taxa_homicidios']

try:
    # Ajustar modelo ARIMA simples
    model = ARIMA(serie, order=(1, 1, 1))
    model_fit = model.fit()
    
    # Fazer previsão para 2025-2026
    forecast = model_fit.forecast(steps=2)
    print(f"  Previsão 2025: {forecast[0]:.1f} homicídios/100k")
    print(f"  Previsão 2026: {forecast[1]:.1f} homicídios/100k")
    print(f"  AIC do modelo: {model_fit.aic:.2f}")
    
except Exception as e:
    print(f"  ⚠️ Erro no modelo ARIMA: {e}")
    print("  Usando regressão linear simples...")
    
    # Regressão linear como fallback
    X_pred = df_brasil['ano'].values.reshape(-1, 1)
    y_pred = df_brasil['taxa_homicidios'].values
    reg = LinearRegression().fit(X_pred, y_pred)
    prev_2025 = reg.predict([[2025]])[0]
    prev_2026 = reg.predict([[2026]])[0]
    print(f"  Previsão 2025 (regressão): {prev_2025:.1f} homicídios/100k")
    print(f"  Previsão 2026 (regressão): {prev_2026:.1f} homicídios/100k")